## **Project Overview**

##### This project asks what share of invasive *E. Coli* isolates reported to EARS-Net were resistant to third-generation cephalosporin, and how that share differd berore and after ERAS-Net moved to EUCAST-only interpretation.

##### From 2020-2025, 13.86% of 986,328 tested isolates were resistant (95% CI 13.79% to 13.92%), which is 1.83 percent point higher than the 2013-2018 (95% CI 11.9% to 12.1%).

##### Resistance is about 1 in 7, and these antibiotics cannot be assumed to work for every invasive infection, although the data covers reported isolates from particiating countries and cannot show that the change in interpretation caused the difference.

### Importing needed libraries and the dataset from the ECDC.



In [30]:
import pandas as pd
import numpy as np
from scipy import stats


In [31]:
# importing dataset

res_data = pd.read_csv("../data/raw/ecdc_surveillance_data_Antimicrobial_resistance.csv")

## Exploratory Data Analysis On Dataset

In [32]:
res_data.head()

,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
0,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,DE,Germany,0.000000000,NaN
1,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,EE,Estonia,0.000000000,NaN
2,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,EL,Greece,7.000000000,NaN
3,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,NL,Netherlands,1.000000000,NaN
4,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,NO,Norway,24.000000000,NaN


In [33]:
res_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3635 entries, 0 to 3634
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   HealthTopic  3635 non-null   str    
 1   Population   3635 non-null   str    
 2   Indicator    3635 non-null   str    
 3   Unit         3635 non-null   str    
 4   Time         3635 non-null   int64  
 5   RegionCode   3635 non-null   str    
 6   RegionName   3635 non-null   str    
 7   NumValue     3635 non-null   str    
 8   TxtValue     0 non-null      float64
dtypes: float64(1), int64(1), str(7)
memory usage: 255.7 KB


In [34]:
res_data.duplicated().any()

np.False_

In [35]:
# Converting NumValue variable from string to float

res_data['NumValue'] = pd.to_numeric(res_data['NumValue'], errors='coerce')

In [36]:
res_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3635 entries, 0 to 3634
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   HealthTopic  3635 non-null   str    
 1   Population   3635 non-null   str    
 2   Indicator    3635 non-null   str    
 3   Unit         3635 non-null   str    
 4   Time         3635 non-null   int64  
 5   RegionCode   3635 non-null   str    
 6   RegionName   3635 non-null   str    
 7   NumValue     3627 non-null   float64
 8   TxtValue     0 non-null      float64
dtypes: float64(2), int64(1), str(6)
memory usage: 255.7 KB


In [37]:
# Investigating which values in the NumValue are null
# There are all in the indicaator 'R - resistant isolates, percentage'. This indicator value isn't needed in our project analysis

res_data[res_data['NumValue'].isna()]

,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
1455,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2000,EE,Estonia,NaN,NaN
1459,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2000,SE,Sweden,NaN,NaN
1511,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2003,CY,Cyprus,NaN,NaN
1702,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2009,SE,Sweden,NaN,NaN
2079,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2022,LI,Liechtenstein,NaN,NaN
2108,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2023,LI,Liechtenstein,NaN,NaN
2138,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2024,LI,Liechtenstein,NaN,NaN
2168,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2025,LI,Liechtenstein,NaN,NaN


### Extracting Needed Indicators For The first investigation _Calculating Confidence Interval of post EUCAST-only adoption (2020-2025)
 The variables needed in other to estimate the proportion of invasive  *E. coli* bacterial isolates that are resistant to Third-generation cephalosporins post EUCAST-only susceptibility interpretation.
 - Resistant Isolates 
 - Total Isolates
 - For a period of 2020 -2025


In [38]:
# Extracting needed variables

study_years = [2020, 2021, 2022, 2023,2024, 2025]

year = res_data[res_data['Time'].isin(study_years)]

needed_indicators= ['R - resistant isolates', 'Total tested isolates']
post_yrs_df = year[year['Indicator'].isin(needed_indicators)]

post_yrs_df.sample(10)



,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
1390,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2023,RO,Romania,189.0,NaN
3496,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2021,ES,Spain,7425.0,NaN
3461,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2020,CY,Cyprus,114.0,NaN
3573,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2023,SI,Slovenia,1639.0,NaN
1416,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2024,NL,Netherlands,817.0,NaN
3550,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2023,CZ,Czechia,3892.0,NaN
3494,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2021,EE,Estonia,929.0,NaN
1298,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2020,NL,Netherlands,498.0,NaN
3622,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2025,LI,Liechtenstein,8.0,NaN
3468,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2020,FI,Finland,5367.0,NaN


In [39]:
# Confirming that both the resistant and total testes isolates have the same count values per year

analysis_gb = post_yrs_df.groupby(['Time', 'Indicator'])['NumValue'].count().reset_index()

analysis_gb

,Time,Indicator,NumValue
0,2020,R - resistant isolates,29
1,2020,Total tested isolates,29
2,2021,R - resistant isolates,29
3,2021,Total tested isolates,29
4,2022,R - resistant isolates,30
5,2022,Total tested isolates,30
6,2023,R - resistant isolates,29
7,2023,Total tested isolates,29
8,2024,R - resistant isolates,30
9,2024,Total tested isolates,30


In [40]:
# Extracting my point estimate

resistant = post_yrs_df[post_yrs_df['Indicator']== 'R - resistant isolates']

post_res = resistant['NumValue'].sum()

total = post_yrs_df[post_yrs_df['Indicator']== 'Total tested isolates']

post_total = total['NumValue'].sum()

print(post_res)
print(post_total)

136656.0
986328.0


In [41]:
post_p = post_res / post_total


print(post_p)


0.13855025914300315


In [42]:
# Chosen confidence level is 0.95 (95%). zscore fot this confidence level is 1.96
# Calculating Standard Error

zscore = 1.96

post_se = np.sqrt((post_p * (1-post_p))/post_total)

post_se

np.float64(0.00034786277930708945)

In [43]:
# Calculating margin of Error

me = zscore * post_se

me

np.float64(0.0006818110474418953)

In [44]:
# The confidence interval of invasive E. Coli to third generation cephalosporins 
# during post EUCAST-only adoption (2020-2025), ranges (13.7% to 13.9%)

lower_confidence, higher_confidence= stats.norm.interval(0.95, post_p, post_se)

print(lower_confidence * 100)
print(higher_confidence * 100)

13.786846062399924
13.923205766200706


### Extracting Needed Indicators For The first investigation _Calculating Confidence Interval of pre EUCAST-only adoption (2013-2018)
 The variables needed in other to estimate the proportion of invasive  *E. coli* bacterial isolates that are resistant to Third-generation cephalosporins pre EUCAST-only susceptibility interpretation.
 - Resistant Isolates 
 - Total Isolates
 - For a period of 2013 -2018

In [45]:
# Extracting needed variables

pre_yrs_study = [2013, 2014, 2015, 2016, 2017, 2018]

pre_yr = res_data[res_data['Time'].isin(pre_yrs_study)]

needed_indicators_pre_yrs= ['R - resistant isolates', 'Total tested isolates']
pre_yrs_df = pre_yr[pre_yr['Indicator'].isin(needed_indicators)]

pre_yrs_df.sample(10)

,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
3399,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2018,BE,Belgium,4644.0,NaN
3365,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2016,SI,Slovenia,1420.0,NaN
3392,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2017,PT,Portugal,6441.0,NaN
3363,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2016,RO,Romania,418.0,NaN
1139,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2015,HR,Croatia,131.0,NaN
3258,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2013,FI,Finland,3720.0,NaN
3264,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2013,IT,Italy,3990.0,NaN
3354,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2016,IT,Italy,5938.0,NaN
3292,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2014,IE,Ireland,2691.0,NaN
3377,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2017,ES,Spain,6027.0,NaN


In [ ]:
# Calculating the proportion of reported resistant invasive E. Coli to third-generation cephalosporin pre EUCAST-only

pre_res = pre_yrs_df[pre_yrs_df['Indicator'] == 'R - resistant isolates']['NumValue'].sum()

pre_total =pre_yrs_df[pre_yrs_df['Indicator'] == 'Total tested isolates']['NumValue'].sum()

pre_p = pre_res / pre_total

print(pre_res)
print(pre_total)
print(pre_p)

81262.0
675790.0
0.12024741413752793


In [47]:
# Calculating standard error for pre EUCAST-only

pre_se = np.sqrt(pre_p * (1 - pre_p) / pre_total)

pre_se

np.float64(0.00039565100063802794)

In [48]:
# Calculating Margin of Error
# Chosen  confidence level is 95% which so we use a zscore of 1.96

pre_moe = 1.96 * pre_se

pre_moe

np.float64(0.0007754759612505348)

In [49]:
# The confidence interval of invasive E. Coli to third generation cephalosporins 
# during pre EUCAST-only adoption (2013-2018), ranges (11.9% to 12.1%)

lower_con, upper_con = stats.norm.interval(0.95, pre_p, pre_se)

print(lower_con * 100)
print(upper_con * 100)

11.947195242583016
12.102287584922571


In [ ]:
# Difference in invasive E Coli resistance proportion between pre and post EUCAST-only

# A gap of 1.83 percentage point, shows that there is aa increase in report of E Coli resistance 
# to third generation cephalosporins since the adoption of EUCAST-only susceptibility interpretation 

difference = (post_p - pre_p) 

print(difference)

0.018302845005475216


#### Investigating if the found difference between pre and post EUCAST-only adoption is real or by chance



In [51]:
# calculating the SE of both periods(pre/post) EUCAST-only adoption

se = np.sqrt(pre_p * (1-pre_p)/pre_total + post_p * (1-post_p)/post_total)

se

np.float64(0.0005268284612026248)

In [52]:
# Calculating confidene interval. chosen confidence level: 0.95


lower, upper = stats.norm.interval(0.95, difference, se)

print(lower * 100)
print(upper * 100)

1.7270280195487413
1.9335409815463018


In [54]:
# Saving my processed dataset

analysis_df = pd.concat([pre_yrs_df, post_yrs_df])

analysis_df.to_csv('analysis_df.csv', index= False)

#### Reported resistance to third generation cephalosporins was 1.83 percentage points higher after EUCAST-only adoption (95% CI 1.73 to 1.93). The interval excludes zero, so the diffference is unlikely to be due to chance, but this comparison cannot show that the change in interpretation caused it.